# bias-correction-divide composite — cx24: Adam ratio: m_hat / (sqrt(v_hat) + eps)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `bias-correction-divide`, `sqrt-eps-stabilize`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "bias-correction-divide"
DD_ATOM_IDS = ["bias-correction-divide", "sqrt-eps-stabilize"]
DD_SUBTOPICS = ["Optimizer: Adam bias-correction divide", "Numerical: sqrt-eps stabilization"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Adam's full parameter update is `p <- p - lr * step` where the step is built from TWO bias-corrected moments and the sqrt-eps denominator:
```
step = m_hat / (sqrt(v_hat) + eps)
     = ( m / (1 - beta1**t) )                         <- bias-correction-divide (atom A)
       / ( sqrt( v / (1 - beta2**t) ) + eps )         <- sqrt-eps-stabilize (atom B)
```
**Both atoms are bias-correction-divide and sqrt-eps-stabilize.** Atom A is invoked TWICE in the equation (once for `m_hat`, once inside the sqrt for `v_hat`), and atom B wraps the result. The composition is the ratio that gives Adam its adaptive, scale-invariant per-coordinate step.

**Why this exact form is scale-invariant.** Multiply `g` by 10. Then `m` scales by 10, and `v` scales by 100, so `sqrt(v_hat)` scales by 10 — the ratio is invariant. That's the whole reason Adam is so robust to gradient magnitude across layers.

**Eps placement matters AT SCALE.** With eps-outside (Adam convention), the ratio at small `v_hat` becomes `m_hat / eps` — a finite-but-large number. With eps-inside (BatchNorm convention), the ratio is `m_hat / sqrt(eps)` — much smaller. Adam wants the MORE AGGRESSIVE step in tiny-`v` regimes, so eps-outside is correct.

**Anatomy.**
```python
def adam_ratio(m, v, beta1, beta2, t, eps):
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    return m_hat / (v_hat.sqrt() + eps)
```

### Composite Exercise — Adam ratio: m_hat / (sqrt(v_hat) + eps)

**Atoms exercised together**: `bias-correction-divide`, `sqrt-eps-stabilize`

Implement `cx24_adam_ratio(m, v, beta1, beta2, t_step, eps)`.

Inputs:
- `m`: first-moment buffer (Tensor).
- `v`: second-moment buffer (Tensor, non-negative elementwise — caller's responsibility).
- `beta1`, `beta2`: floats.
- `t_step`: int >= 1.
- `eps`: float (typically 1e-8).

Steps:
1. `m_hat = m / (1 - beta1 ** t_step)` (atom: bias-correction-divide).
2. `v_hat = v / (1 - beta2 ** t_step)` (atom: bias-correction-divide — second use).
3. `denom = sqrt(v_hat) + eps` (atom: sqrt-eps-stabilize — eps OUTSIDE).
4. Return `m_hat / denom` (a fresh tensor, same shape as `m`).

Do not mutate `m` or `v`. Cross-check the full-step equation vs `torch.optim.Adam`'s reference, since this is the EXACT ratio Adam multiplies by `-lr`.

Tests verify:
- `m=0, v=0 -> ratio=0` (zero numerator, finite denominator from eps).
- Sign of the ratio matches sign of `m` elementwise (denominator is always positive).
- Scale-invariance: multiplying `(m, v)` by `(c, c**2)` leaves the ratio unchanged.
- Cross-check: replicates Adam's update direction (`-step`) from `torch.optim.Adam`.
- Eps placement: with `m=1, v=0`, ratio == `1 / eps` (eps outside), NOT `1 / sqrt(eps)`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx24_adam_ratio(m: Tensor, v: Tensor, beta1: float, beta2: float, t_step: int, eps: float) -> Tensor:
    """Return m_hat / (sqrt(v_hat) + eps), the Adam step direction."""
    raise NotImplementedError

def _test_cx24():
    # Case A: m=0, v=0 — ratio should be 0 (numerator is 0, denominator is eps>0).
    m = t.zeros(3)
    v = t.zeros(3)
    r = cx24_adam_ratio(m, v, 0.9, 0.999, t_step=1, eps=1e-8)
    assert tuple(r.shape) == (3,)
    assert t.allclose(r, t.zeros(3), atol=1e-12), f'ratio with zero m and v must be 0; got {r}'

    # Case B: closed-form step 1. Suppose g was constant; then after 1 step:
    #   m = (1 - beta1) * g, v = (1 - beta2) * g**2
    #   m_hat = g, v_hat = g**2, sqrt(v_hat) = |g|.
    #   ratio = g / (|g| + eps) ~ sign(g) for |g| >> eps.
    beta1 = 0.9
    beta2 = 0.999
    g = t.tensor([1.0, -2.0, 0.5, -0.1])
    m = (1 - beta1) * g
    v = (1 - beta2) * g.pow(2)
    r = cx24_adam_ratio(m, v, beta1, beta2, t_step=1, eps=1e-8)
    expected = g / (g.abs() + 1e-8)
    assert t.allclose(r, expected, atol=1e-5), f'closed-form mismatch: got {r}, expected {expected}'
    # Sign sanity.
    assert t.equal(r.sign(), g.sign()), f'ratio sign should match m sign; got signs {r.sign()}'

    # Case C: scale-invariance — scaling (m, v) by (c, c**2) is a no-op on the ratio.
    m_a = t.tensor([0.3, -0.7, 0.2])
    v_a = t.tensor([0.5, 0.5, 0.5])
    c = 100.0
    r_a = cx24_adam_ratio(m_a, v_a, 0.9, 0.999, t_step=10, eps=1e-12)  # tiny eps to expose scale-invariance.
    r_b = cx24_adam_ratio(m_a * c, v_a * (c ** 2), 0.9, 0.999, t_step=10, eps=1e-12)
    assert t.allclose(r_a, r_b, atol=1e-5), (
        f'Adam ratio must be scale-invariant under (m, v) -> (c*m, c**2*v); '
        f'got r_a={r_a}, r_b={r_b}'
    )

    # Case D: eps placement. m=1, v=0 — ratio = 1 / (0 + eps) = 1/eps.
    m1 = t.tensor([1.0])
    v0 = t.tensor([0.0])
    r_eps = cx24_adam_ratio(m1, v0, 0.9, 0.999, t_step=10000, eps=1e-8)  # large t so m_hat ~ m.
    # m_hat ~ 1.0, v_hat ~ 0.0, denom = sqrt(0) + 1e-8 = 1e-8. ratio = 1e8.
    expected_huge = 1.0 / 1e-8
    assert abs(r_eps.item() - expected_huge) / expected_huge < 0.01, (
        f'eps placement: expected ratio ~{expected_huge:.2e} (eps OUTSIDE sqrt); got {r_eps.item():.2e} — '
        f'if you got ~1e4 you wrote sqrt(v_hat + eps) instead of sqrt(v_hat) + eps.'
    )

    # Case E: inputs not mutated.
    m_in = t.tensor([0.5, -0.5])
    v_in = t.tensor([1.0, 4.0])
    snap_m, snap_v = m_in.clone(), v_in.clone()
    _ = cx24_adam_ratio(m_in, v_in, 0.9, 0.999, t_step=3, eps=1e-8)
    assert t.equal(m_in, snap_m), 'm was mutated'
    assert t.equal(v_in, snap_v), 'v was mutated'

    # Case F: cross-check vs torch.optim.Adam — the ratio is the parameter step magnitude / lr.
    # We replicate Adam's first step by hand and compare with the actual delta on p.
    t.manual_seed(23)
    p_ref = t.nn.Parameter(t.randn(5))
    p_before = p_ref.detach().clone()
    g_ref = t.randn(5)
    lr = 0.05
    eps = 1e-8
    opt = t.optim.Adam([p_ref], lr=lr, betas=(0.9, 0.999), eps=eps)
    p_ref.grad = g_ref.clone()
    opt.step()
    # Adam: p <- p - lr * ratio, so ratio_observed = (p_before - p_after) / lr.
    ratio_observed = (p_before - p_ref.detach()) / lr
    # Reconstruct what cx24 returns at step 1 starting from m=0, v=0.
    m0 = (1 - 0.9) * g_ref
    v0 = (1 - 0.999) * g_ref.pow(2)
    ratio_ours = cx24_adam_ratio(m0, v0, 0.9, 0.999, 1, eps)
    assert t.allclose(ratio_ours, ratio_observed, atol=1e-5), (
        f'cx24 ratio disagrees with torch.optim.Adam step direction; '
        f'max err = {(ratio_ours - ratio_observed).abs().max().item():.2e}'
    )

    # Case G: shape preserved on multi-dim.
    m_2d = t.randn(3, 4)
    v_2d = t.rand(3, 4)
    r_2d = cx24_adam_ratio(m_2d, v_2d, 0.9, 0.999, 5, 1e-8)
    assert tuple(r_2d.shape) == (3, 4)
    _dd_passed.add('cx24')

_test_cx24()

<details><summary>Show solution — cx24</summary>

```python
def cx24_adam_ratio(m, v, beta1, beta2, t_step, eps):
    # Atom A (bias-correction-divide): applied to m...
    m_hat = m / (1.0 - beta1 ** t_step)
    # ...and to v (same atom, second invocation, using beta2).
    v_hat = v / (1.0 - beta2 ** t_step)
    # Atom B (sqrt-eps-stabilize): Adam convention — eps OUTSIDE the sqrt.
    denom = v_hat.sqrt() + eps
    return m_hat / denom
```

**This is the load-bearing line of Adam.** Everything else (m/v EMAs, beta defaults, lr schedule, weight decay) is decoration around this ratio. Getting the ratio's structure wrong is what makes a from-scratch Adam train slower than `torch.optim.Adam`.

**Two atoms, three operations.** The bias-correction-divide atom is invoked TWICE in the same expression — once on `m`, once on `v` — because both moments need de-biasing at the same step `t`. The sqrt-eps-stabilize atom wraps the v-side result. This is the most multiply-invoked atom composition in the entire Adam step.

**Scale invariance is the diagnostic.** If your ratio depends on the absolute magnitude of `g` (not just its direction relative to history), you've broken the bias correction or moved eps inside the sqrt. Case C is the cleanest invariant: scaling `(m, v)` by `(c, c**2)` MUST leave the ratio invariant (for `eps << sqrt(v_hat)`).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx24'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx24',
        'subtopics': ["Optimizer: Adam bias-correction divide", "Numerical: sqrt-eps stabilization"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()